# Can a model that has never seen an MRI read one? CLIP zero-shot vs CoOp

### The idea in plain words

**CLIP** is a model trained by OpenAI on 400 million internet pictures and their captions.
It learned to put a picture and a sentence that describes it *close together* in a shared
space of numbers. Because of that, you can classify a picture without any training at all:
write one sentence per class ("a photo of a dog", "a photo of a cat"), and ask CLIP which
sentence the picture is closest to. That is called **zero-shot** classification: zero
training examples.

But CLIP learned from everyday photos, not brain scans. So our first question is:
**how well does zero-shot CLIP do on MRI scans?** (Spoiler: badly, close to guessing.)

**CoOp** (Context Optimization, Zhou et al. 2022) is a fix that keeps CLIP completely
frozen and only learns the *words* of the prompt. Instead of us writing "a photo of a", CoOp
learns a few "soft words" (vectors of numbers that don't have to be real words) that go in
front of each class name. Only those few vectors are trained. Second question:
**how far can a handful of learned prompt vectors lift the accuracy?**

We use the same brain tumor MRI dataset and the same leakage-free split as the main
project, so the numbers are directly comparable to the CNN and DCFormer results.

Every cell is commented line by line. Run them top to bottom on a GPU runtime.

## 1. Setup

In [ ]:
# open_clip is the library that gives us CLIP. Install it (Colab doesn't have it by default).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)

In [ ]:
import os, random, json
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import open_clip                                   # the CLIP library
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Data: the same leakage-free split as the main project

This is the same code as the brain tumor notebook, condensed. Quick reminder of what it does:
1. Walk the four class folders and record every image's path and label.
2. Fingerprint every image (dHash) so exact copies get the same fingerprint.
3. Split by fingerprint group, 70/15/15, so no scan appears on two sides.

In [ ]:
if os.environ.get("DATA_DIR"):
    path = os.environ["DATA_DIR"]
else:
    import kagglehub
    kagglehub.login()
    path = kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")

CLASSES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]

def collect_images(root_dir, split_name):
    records = []
    for label_idx, class_name in enumerate(CLASSES):
        class_dir = os.path.join(root_dir, class_name)
        for fname in sorted(os.listdir(class_dir)):
            records.append({"path": os.path.join(class_dir, fname), "label": label_idx, "split": split_name})
    return records

all_records = (collect_images(os.path.join(path, "Training"), "train")
               + collect_images(os.path.join(path, "Testing"), "test"))

def image_fingerprint(img_path, hash_size=8):
    img = Image.open(img_path).convert("L").resize((hash_size + 1, hash_size), Image.LANCZOS)
    px = np.asarray(img, dtype=np.int16)
    return "".join("1" if b else "0" for b in (px[:, 1:] > px[:, :-1]).flatten())

for rec in all_records:
    rec["fp"] = image_fingerprint(rec["path"])
groups = defaultdict(list)
for rec in all_records:
    groups[rec["fp"]].append(rec)
for gid, recs in enumerate(groups.values()):
    for rec in recs:
        rec["group"] = gid

def grouped_stratified_split(records, train_frac=0.70, val_frac=0.15, seed=SEED):
    rng = random.Random(seed)
    group_label = {}
    for gid in {r["group"] for r in records}:
        group_label[gid] = Counter(r["label"] for r in records if r["group"] == gid).most_common(1)[0][0]
    split_of_group = {}
    for cls in range(len(CLASSES)):
        gids = [g for g, l in group_label.items() if l == cls]
        rng.shuffle(gids)
        n_train, n_val = int(len(gids) * train_frac), int(len(gids) * val_frac)
        for i, g in enumerate(gids):
            split_of_group[g] = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
    for rec in records:
        rec["my_split"] = split_of_group[rec["group"]]
    return records

all_records = grouped_stratified_split(all_records)
seen, clean = set(), []
for rec in all_records:
    if rec["group"] not in seen:
        seen.add(rec["group"]); clean.append(rec)
splits = {s: [r for r in clean if r["my_split"] == s] for s in ["train", "val", "test"]}
for s, recs in splits.items():
    print(f"{s:5s}: {len(recs):4d} images")

## 3. Load CLIP

We use **ViT-B/32**, the smallest standard CLIP. "ViT" = Vision Transformer (the image half),
"B/32" = base size, image cut into 32x32 patches. The download is about 600 MB.

`preprocess` is the exact resize/crop/normalize recipe CLIP was trained with. We must use it,
because CLIP expects images in that precise form.

We **freeze** CLIP: `requires_grad_(False)` tells PyTorch never to update its weights.

In [ ]:
# PRETRAINED is "openai" for the real weights. Tests set it to None to skip the download.
PRETRAINED = os.environ.get("CLIP_PRETRAINED", "openai") or None

model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer("ViT-B-32")   # turns a sentence into a list of token numbers
model = model.to(device).eval()                    # eval mode: no dropout, fixed behaviour
model.requires_grad_(False)                        # freeze every weight

print("Image feature size:", model.visual.output_dim)
print("Trainable parameters in CLIP right now:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Turn every image into a CLIP feature vector, once

CLIP's image encoder is frozen, so the vector it produces for a given scan never changes.
We compute it once per image and save it. After this, both zero-shot and CoOp only work with
these vectors, which makes everything that follows fast.

Each vector has 512 numbers, and we **normalize** it to length 1 so that comparing two
vectors is just a dot product (the cosine of the angle between them).

In [ ]:
class CLIPImages(Dataset):
    def __init__(self, records): self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        rec = self.records[i]
        img = Image.open(rec["path"]).convert("RGB")   # CLIP expects 3 colour channels
        return preprocess(img), rec["label"]

@torch.no_grad()                                     # no gradients: just computing, not training
def encode_split(records):
    loader = DataLoader(CLIPImages(records), batch_size=64, shuffle=False, num_workers=2)
    feats, labels = [], []
    for xb, yb in loader:
        f = model.encode_image(xb.to(device))        # (B, 512)
        feats.append(F.normalize(f, dim=-1).cpu())   # length-1 vectors
        labels.append(yb)
    return torch.cat(feats), torch.cat(labels)

features = {s: encode_split(recs) for s, recs in splits.items()}
for s, (f, y) in features.items():
    print(f"{s:5s}: features {tuple(f.shape)}, labels {tuple(y.shape)}")

## 5. Zero-shot CLIP

For each class we write a sentence, encode it with CLIP's text encoder, and normalize it.
To classify an image: dot its vector with each of the 4 text vectors, and pick the biggest.

CLIP's text and image spaces are only aligned up to a scale, so the dot products are
multiplied by `logit_scale` (about 100) before softmax. That is standard CLIP behaviour.

In [ ]:
CLASS_WORDS = ["glioma tumor", "meningioma tumor", "no tumor", "pituitary tumor"]   # human-readable
prompt_templates = [
    "a photo of a {}.",
    "an MRI scan of a brain with {}.",
    "a brain MRI showing {}.",
]

@torch.no_grad()
def zero_shot_text_features(template):
    sentences = [template.format(w) for w in CLASS_WORDS]
    tokens = tokenizer(sentences).to(device)            # (4, 77) token ids, padded to 77
    t = model.encode_text(tokens)                       # (4, 512)
    return F.normalize(t, dim=-1).cpu()

def evaluate_logits(logits, labels):
    probs = logits.softmax(dim=-1).numpy()
    preds = probs.argmax(1)
    return {"accuracy": accuracy_score(labels, preds),
            "auc": roc_auc_score(labels, probs, multi_class="ovr"),
            "confusion": confusion_matrix(labels, preds, labels=list(range(4)))}

test_f, test_y = features["test"]
logit_scale = model.logit_scale.exp().item()
zs_results = {}
for tpl in prompt_templates:
    text_f = zero_shot_text_features(tpl)
    logits = logit_scale * test_f @ text_f.T            # (N_test, 4) similarity scores
    r = evaluate_logits(logits, test_y.numpy())
    zs_results[tpl] = r
    print(f"zero-shot | {tpl:40s} | acc {r['accuracy']*100:5.1f}% | AUC {r['auc']:.3f}")
print(f"\nChance level with 4 classes is 25%.")

## 6. CoOp: learn the prompt

A CLIP prompt goes through the text encoder as a sequence of 77 **token embeddings**
(512-number vectors), one per word piece. For the sentence `"X X X X glioma tumor."`, CoOp:

1. Replaces the embeddings of the 4 placeholder `X` tokens with 4 **learnable vectors**
   (the "context"). These are the only parameters we train, 4 x 512 = 2,048 numbers.
2. Keeps the class-name tokens, the start token and the end token exactly as CLIP embeds them.
3. Runs the whole sequence through CLIP's frozen text transformer to get a class vector.

Training is ordinary classification: cross-entropy between image-text similarities and the
true label. Gradients flow back through the frozen transformer into the 4 context vectors
only. We train on the training split and pick the best epoch on validation.

In [ ]:
N_CTX = 4                     # number of learnable context "words"
CTX_INIT = "a photo of a"     # start the context from these real words (helps convergence)

class PromptLearner(nn.Module):
    def __init__(self, clip_model, classnames, n_ctx=N_CTX, ctx_init=CTX_INIT):
        super().__init__()
        dtype = clip_model.token_embedding.weight.dtype
        emb_dim = clip_model.token_embedding.weight.shape[1]           # 512
        n_cls = len(classnames)

        # --- 1. learnable context vectors ---
        if ctx_init:
            init_tokens = tokenizer([ctx_init]).to(device)             # (1, 77)
            with torch.no_grad():
                init_emb = clip_model.token_embedding(init_tokens)     # (1, 77, 512)
            ctx_vectors = init_emb[0, 1:1 + n_ctx, :].clone()           # skip the start token
        else:
            ctx_vectors = torch.randn(n_ctx, emb_dim, dtype=dtype, device=device) * 0.02
        self.ctx = nn.Parameter(ctx_vectors)                            # <- the trainable part

        # --- 2. frozen pieces: start token, class name tokens, end token ---
        placeholder = " ".join(["X"] * n_ctx)
        prompts = [f"{placeholder} {name}." for name in classnames]
        tokens = tokenizer(prompts).to(device)                          # (n_cls, 77)
        with torch.no_grad():
            emb = clip_model.token_embedding(tokens)                    # (n_cls, 77, 512)
        # register_buffer = stored with the model but NOT trained
        self.register_buffer("prefix", emb[:, :1, :])                   # start-of-text token
        self.register_buffer("suffix", emb[:, 1 + n_ctx:, :])           # class name, ".", end, padding
        self.register_buffer("tokens", tokens)                          # needed to find the end token
        self.n_cls = n_cls

    def forward(self):
        ctx = self.ctx.unsqueeze(0).expand(self.n_cls, -1, -1)          # same context for every class
        return torch.cat([self.prefix, ctx, self.suffix], dim=1)        # (n_cls, 77, 512)

class TextEncoder(nn.Module):
    """CLIP's text encoder, but starting from embeddings instead of token ids."""
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.attn_mask = clip_model.attn_mask

    def forward(self, prompt_emb, tokens):
        x = prompt_emb + self.positional_embedding                       # add position info
        x = self.transformer(x, attn_mask=self.attn_mask)
        x = self.ln_final(x)
        # take the vector at the end-of-text token (the highest token id in each row)
        x = x[torch.arange(x.shape[0]), tokens.argmax(dim=-1)]
        return x @ self.text_projection                                  # (n_cls, 512)

class CoOp(nn.Module):
    def __init__(self, clip_model, classnames):
        super().__init__()
        self.prompt_learner = PromptLearner(clip_model, classnames)
        self.text_encoder = TextEncoder(clip_model)
        self.logit_scale = clip_model.logit_scale

    def forward(self, image_features):                                   # image_features: (B, 512), normalized
        text_features = self.text_encoder(self.prompt_learner(), self.prompt_learner.tokens)
        text_features = F.normalize(text_features, dim=-1)
        return self.logit_scale.exp() * image_features @ text_features.T  # (B, 4)

coop = CoOp(model, CLASS_WORDS).to(device)
trainable = [p for p in coop.parameters() if p.requires_grad]
print("Trainable parameters:", sum(p.numel() for p in trainable))        # should be N_CTX * 512

### Train CoOp

The image features are already computed, so each epoch is just a few matrix multiplications.
SGD with a cosine schedule, as in the CoOp paper.

In [ ]:
EPOCHS = int(os.environ.get("EPOCHS", 50))
train_f, train_y = features["train"]
val_f, val_y = features["val"]

optimizer = torch.optim.SGD(trainable, lr=2e-3, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def batches(f, y, bs=64, shuffle=True):
    idx = torch.randperm(len(y)) if shuffle else torch.arange(len(y))
    for i in range(0, len(y), bs):
        j = idx[i:i + bs]
        yield f[j].to(device), y[j].to(device)

@torch.no_grad()
def coop_accuracy(f, y):
    coop.eval()
    logits = coop(f.to(device)).cpu()
    return (logits.argmax(1) == y).float().mean().item()

best_val, best_ctx, history = 0.0, None, []
for epoch in range(1, EPOCHS + 1):
    coop.train()
    total = 0.0
    for xb, yb in batches(train_f, train_y):
        loss = F.cross_entropy(coop(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(yb)
    scheduler.step()
    tr_acc, va_acc = coop_accuracy(train_f, train_y), coop_accuracy(val_f, val_y)
    history.append((total / len(train_y), tr_acc, va_acc))
    if va_acc > best_val:
        best_val, best_ctx = va_acc, coop.prompt_learner.ctx.detach().clone()
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:3d} | loss {total/len(train_y):.3f} | train acc {tr_acc:.3f} | val acc {va_acc:.3f}")

coop.prompt_learner.ctx.data.copy_(best_ctx)       # restore the best context vectors
print(f"best val acc: {best_val:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([h[1] for h in history], label="train acc")
plt.plot([h[2] for h in history], label="val acc")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title("CoOp training"); plt.legend(); plt.show()

## 7. Final comparison on the clean test set

In [ ]:
with torch.no_grad():
    coop.eval()
    coop_logits = coop(test_f.to(device)).cpu()
coop_res = evaluate_logits(coop_logits, test_y.numpy())

best_zs_tpl = max(zs_results, key=lambda t: zs_results[t]["accuracy"])
best_zs = zs_results[best_zs_tpl]

print(f"{'Method':34s} | accuracy | AUC")
print(f"{'Chance (4 classes)':34s} |   25.0%  |  0.500")
print(f"{'Zero-shot CLIP (best prompt)':34s} |  {best_zs['accuracy']*100:5.1f}%  |  {best_zs['auc']:.3f}")
print(f"{'CoOp (' + str(N_CTX) + ' learned context vectors)':34s} |  {coop_res['accuracy']*100:5.1f}%  |  {coop_res['auc']:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
short = [c.replace("_tumor", "") for c in CLASSES]
for ax, (name, r) in zip(axes, [("Zero-shot CLIP", best_zs), ("CoOp", coop_res)]):
    cm = r["confusion"]
    ax.imshow(cm, cmap="Blues")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels(short, rotation=30)
    ax.set_yticks(range(4)); ax.set_yticklabels(short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
plt.tight_layout(); plt.show()

## 8. What did CoOp learn? Nearest real words to each context vector

The learned vectors don't have to be words, but we can ask which real token embedding each
one is closest to. Usually the result is not readable, which is itself an interesting finding
from the CoOp paper: the best prompts are not human language.

In [ ]:
with torch.no_grad():
    vocab = model.token_embedding.weight                       # (49408, 512): every token CLIP knows
    ctx = coop.prompt_learner.ctx                              # (N_CTX, 512)
    dists = torch.cdist(ctx, vocab)                            # distance from each ctx vector to each token
    nearest = dists.argmin(dim=1).tolist()
decoder = {v: k for k, v in tokenizer.encoder.items()} if hasattr(tokenizer, "encoder") else None
words = [decoder.get(i, str(i)).replace("</w>", "") if decoder else str(i) for i in nearest]
print("Nearest real tokens to the learned context:", words)

## 9. Save results

In [ ]:
summary = {
    "zero_shot": {tpl: {"accuracy_pct": round(r["accuracy"] * 100, 1), "auc": round(r["auc"], 3)}
                  for tpl, r in zs_results.items()},
    "coop": {"n_ctx": N_CTX, "trainable_params": sum(p.numel() for p in trainable),
             "accuracy_pct": round(coop_res["accuracy"] * 100, 1), "auc": round(coop_res["auc"], 3)},
    "test_images": int(len(test_y)),
}
with open("results_clip_coop.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))